# Persisting memory across Strands Agents sessions

In this example you will learn how to persist memory across different sessions in your Strands Agents. 

We will use the use case of an agent that does web search using a `duckduckgo` search API.

In this notebook, we will:
- Explore the capabilities of a memory-powered Strands agent.
- Learn how to store, retrieve, and list memories.
- Understand how to perform web searches via the agent.
- Interact with the agent in an interactive loop.


### Usage Examples

Storing memories:
```
Remember that I prefer tea over coffee
```

Retrieving memories:
```
What do I prefer to drink?
```

Listing all memories:
```
Show me everything you remember about me
```

### Tips for Memory Usage

- Be explicit when asking the agent to remember information
- Use specific queries to retrieve relevant memories
- Memory persistence enables more natural and contextual conversations

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account and AWS credentials configured in the environment
* Anthropic Claude 3.7 enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket and Amazon DynamoDB

Let's now install the requirement packages for our Strands Agent

In [1]:
# Install the required packages
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [mem0ai]2m5/7 [qdrant-client]


In [2]:

# Import Required Libraries
import os
from strands import Agent, tool
from strands.models import bedrock
from strands_tools import mem0_memory

from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException

bedrock.DEFAULT_BEDROCK_MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0" #Optional: Set a default model for Bedrock

## Mem0 Configuration

### Memory Backend Options

The Mem0 Memory Tool supports three different backend configurations:

1. **[OpenSearch](https://aws.amazon.com/opensearch-service/features/serverless/)** (Recommended for AWS environments):
   - Requires AWS credentials and OpenSearch configuration
   - Set `OPENSEARCH_HOST` and optionally `AWS_REGION` (defaults to us-west-2)

2. **[FAISS]((https://faiss.ai/index.html))** (Default for local development):
   - Uses FAISS as the local vector store backend
   - Requires faiss-cpu package for local vector storage
   - No additional configuration needed

3. **Mem0 Platform**:
   - Uses the [Mem0 Platform API](https://docs.mem0.ai/platform/quickstart) for memory management
   - Requires a Mem0 API key : `MEM0_API_KEY` in the environment variables


### Environment Configuration

| Environment Variable | Description | Default | Required For |
|---------------------|-------------|----------|--------------|
| OPENSEARCH_HOST | OpenSearch Serverless Host URL | None | OpenSearch |
| AWS_REGION | AWS Region for OpenSearch | us-west-2 | OpenSearch |
| MEM0_API_KEY | Mem0 Platform API key | None | Mem0 Platform |
| DEV | Enable development mode | false | All modes |


For the scope of this lab, we can use 2 options as a backend for memory management:
### Option 1. [Opensearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/) 

This will be our setup architecture for AOSS: 

<div style="text-align:left">
    <img src="images/arch_AOSS.png" width="65%" />
</div>



In [3]:
# You can manually define your Opensearch Host 
#os.environ["OPENSEARCH_HOST"] = "<your-opensearch-host>.<region>.aoss.amazonaws.com"

In [4]:
# OR - Run the script to Create Opensearch Serverless resource in your AWS Account
!bash prereqs/deploy_OSS.sh

deploying opensearch Serverless ...


/opt/conda/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=7524) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Configuration loaded: {'opensearch_collection_name': 'memory-store', 'opensearch_description': 'Memory vector store'}
Creating OpenSearch Serverless collection: memory-store
Waiting for policies to propagate...
Creating collection...

Collection successfully created:
[{'id': '39o4sbp9k495db17xtpl', 'name': 'memory-store', 'status': 'ACTIVE', 'type': 'VECTORSEARCH', 'description': 'Memory vector store', 'arn': 'arn:aws:aoss:us-west-2:577919262841:collection/39o4sbp9k495db17xtpl', 'kmsKeyArn': 'auto', 'standbyReplicas': 'ENABLED', 'createdDate': 1753909950116, 'lastModifiedDate': 1753909973885, 'collectionEndpoint': 'https://39o4sbp9k495db17xtpl.us-west-2.aoss.amazonaws.com', 'dashboardEndpoint': 'https://39o4sbp9k495db17xtpl.us-west-2.aoss.amazonaws.com/_dashboards'}]
Environment variables saved to /home/sagemaker-user/samples/01-tutorials/01-fundamentals/07-memory-persistent-agents/.env
OpenSearch Serverless collection setup completed!

OPENSEARCH_HOST: 39o4sbp9k495db17xtpl.us-west-2.a

In [5]:
# Option 1: Opensearch Serverless
from dotenv import load_dotenv
load_dotenv() # take Opensearch environment variables

True


### Option 2 [Mem0 Platform](https://docs.mem0.ai/platform):

#### [NOTE]: This is not Needed if you have already deployed the Opensearch Serverless option.

As an alternative, you can create Mem0 API keys by following the steps [here](https://docs.mem0.ai/platform/quickstart#2-api-key-setup) and add it as an environment variable **MEM0_API_KEY**.
This would be the architecture of the setup for Mem0 Platform:

<div style="text-align:left">
    <img src="images/arch_mem0.png" width="65%" />
</div>


To enable the agent's functionality, we need to configure environment variables for AWS credentials and OpenSearch / Mem0 Platform. These variables are used for memory storage and retrieval.

In [6]:
# Option 2: Mem0 API key

#os.environ["MEM0_API_KEY"] = "<your-mem0-api-key>"

## Define System Prompt

The `SYSTEM_PROMPT` variable defines the behavior and capabilities of the memory agent. This prompt guides the agent to provide personalized responses based on stored memories and perform web searches when necessary.

In [7]:
# Define a focused system prompt for memory operations
SYSTEM_PROMPT = """You are a helpful personal assistant for a user. Your task is to assist the user by providing personalized responses based on their history. 

Capabilities:
- You can store information using the mem0_memory tool (action="store").
- You can retrieve relevant memories using the mem0_memory tool (action="retrieve").
- You can use duckduckgo_search to find information on the web.

Key Rules:
- Be conversational and natural in your responses.
- Always retrieve memories before responding to the user and use them to inform your response.
- Store any new user information and user preferences in mem0_memory.
- Only share relevant information.
- Politely indicate when you don't have the information.
"""

## Define Web Search Tool

The `websearch` tool using [Duckduckgo Search API](https://github.com/deedy5/duckduckgo_search) function allows the agent to perform web searches. This function handles exceptions and returns search results or appropriate error messages.

In [8]:
@tool
def websearch(
    keywords: str,
    region: str = "us-en",
    max_results: int | None = None,
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

## Create Memory Agent

We will now initialize the memory-focused agent using the defined tools and system prompt. The Strands agent is capable of:
1. Storing and retrieving memories based on context. It uses memory to create more personalized and contextual AI interactions.
2. Performing web searches using DuckDuckGo to give updated information.

In [9]:
# Create an agent with memory, websearch tool
USER_ID = "new_user" # Replace with actual user ID

memory_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # Optional: Specify the model ID
    tools=[mem0_memory, websearch],
)

## Demonstrate Memory Operations

The following examples demonstrate how to store, retrieve, and list memories using the memory agent.

- **store**: Save important information for later retrieval
  - Store user preferences
  - Remember important facts
  - Maintain conversation context

- **retrieve**: Access relevant memories based on queries
  - Find previously stored information
  - Provide personalized responses based on user history

- **list**: View all stored memories
  - See what information has been retained
  - Audit stored memories

In [10]:
# Store initial memories to demonstrate retrieval
memory_agent.tool.mem0_memory(
    action="store", content=f"The user's name is {USER_ID}.", user_id=USER_ID
)
memory_agent.tool.mem0_memory(
    action="store", 
    content="I like to drink tea more than coffee.", 
    user_id=USER_ID
)

/opt/conda/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
Creating index mem0_memories, it might take 1-2 minutes...
Creating index mem0migrations, it might take 1-2 minutes...
Error retrieving vector 10985f40-6409-4202-83d1-3a0313f4e5a4: NotFoundError(404, 'index_not_found_exception', 'no such index [mem0migrations]', mem0migrations, index_or_alias)


╭─────────────────────────────────────────── Memory for user new_user ────────────────────────────────────────────╮
│ The user's name is new_user.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/opt/conda/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()


╭───────────────────────────────────────────────── Memory Stored ─────────────────────────────────────────────────╮
│                           Memory Stored                                                                         │
│ ┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                              │
│ ┃ Operation ┃ Content                                            ┃                                              │
│ ┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                              │
│ │ ADD       │ Name is new_user                                   │                                              │
│ └───────────┴────────────────────────────────────────────────────┘                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Memory for user new_user ────────────────────────────────────────────╮
│ I like to drink tea more than coffee.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Memory Stored ─────────────────────────────────────────────────╮
│                           Memory Stored                                                                         │
│ ┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                              │
│ ┃ Operation ┃ Content                                            ┃                                              │
│ ┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                              │
│ │ ADD       │ Prefers tea over coffee                            │                                              │
│ └───────────┴────────────────────────────────────────────────────┘                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_550905595',
 'status': 'success',
 'content': [{'text': '[\n  {\n    "id": "dd234404-57bf-4f21-b1df-b8b92cd94c64",\n    "memory": "Prefers tea over coffee",\n    "event": "ADD"\n  }\n]'}]}

In [11]:
# Retrieve memories
retrieved_memories = memory_agent.tool.mem0_memory(
    action="retrieve", query="What is the user's name?", user_id=USER_ID
)
print("Retrieved Memories:", retrieved_memories)

╭────────────────────────────────────────────────── No Matches ───────────────────────────────────────────────────╮
│ No memories found matching the query.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Retrieved Memories: {'toolUseId': 'tooluse_mem0_memory_905174049', 'status': 'success', 'content': [{'text': '[]'}]}


In [12]:
# Retrieve memories about preferences
memory_agent.tool.mem0_memory(
    action="retrieve",
    query="What are the my drink preferences?",
    user_id=USER_ID
)

╭────────────────────────────────────────────────── No Matches ───────────────────────────────────────────────────╮
│ No memories found matching the query.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_994113215',
 'status': 'success',
 'content': [{'text': '[]'}]}

In [13]:
# Ask the agent a question
response = memory_agent("What are the events happening in the New York today?")
print(response)

I'll search for events happening in New York today for you.
Tool #1: mem0_memory


╭────────────────────────────────────────────────── No Matches ───────────────────────────────────────────────────╮
│ No memories found matching the query.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Tool #2: websearch

Tool #3: mem0_memory


╭─────────────────────────────────────────── Memory for user new_user ────────────────────────────────────────────╮
│ The user asked about events happening in New York today, showing interest in New York activities.               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Memory Stored ─────────────────────────────────────────────────╮
│                           Memory Stored                                                                         │
│ ┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                              │
│ ┃ Operation ┃ Content                                            ┃                                              │
│ ┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                              │
│ │ ADD       │ Interested in events happening in New York today   │                                              │
│ └───────────┴────────────────────────────────────────────────────┘                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on my search, here are some events happening in New York today:

1. There are various free events, concerts, and art exhibitions throughout NYC today. Time Out New York mentions "kick-ass fun, live shows, art around the city and more."

2. Eventbrite has a full listing of various activities and events happening today in New York City that you can browse by interest.

3. New York City has several entertainment options today including star-studded concerts, Broadway shows, family shows, comedy performances, ballet, and opera performances.

4. Sports fans can find sporting events today, as NYC is home to hockey, basketball, and baseball teams.

5. Fever app is also promoting various experiences in New York today including concerts, exhibitions, museum events, and restaurant experiences.

For more specific recommendations or to narrow down by your interests (such as music, art, sports, etc.), I'd be happy to do a more targeted search. Would you like more details about any particular 

In [14]:

# List all stored memories
print("All Stored Memories:")
all_memories = memory_agent.tool.mem0_memory(
    action="list", user_id=USER_ID
)

All Stored Memories:


╭────────────────────────────────────────────────── No Memories ──────────────────────────────────────────────────╮
│ No memories found.                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Interactive Agent Usage

Finally, we provide an interactive loop for users to interact with the memory agent. Users can store new memories, retrieve existing ones, or list all stored memories.

To test interactive usage: 
1. Install the requirements: `pip install -r requirements.txt`
1. Run the python file `personal_agent_with_memory.py`. 

## Conclusion

This notebook demonstrates how to create a personal agent with memory capabilities using the Strands framework. The agent can:

1. Store information about the user
2. Retrieve relevant memories based on context
3. Search the web for additional information
4. Provide personalized responses

By combining these capabilities, the agent can maintain context across conversations and provide more personalized assistance over time.

### Cleanup

Run this bash script to clean up the Opensearch Serverless resources. You don't need to run this if you used the "MEM0_PLATFORM_API".

In [15]:
!sh prereqs/cleanup_OSS.sh

Removing Opensearch Serverless resources ...


/opt/conda/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=7524) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Configuration loaded: {'opensearch_collection_name': 'memory-store', 'opensearch_description': 'Memory vector store'}
Deleting OpenSearch Serverless collection: memory-store
Found collection ID: 39o4sbp9k495db17xtpl
Collection memory-store deleted successfully!
Waiting for collection deletion to complete...
Access policy memory-store-ap-1753909944 deleted successfully!
Network policy memory-store-np-1753909944 deleted successfully!
Encryption policy memory-store-sp-1753909944 deleted successfully!
OpenSearch Serverless resources cleanup completed!
Deleted SSM parameter: memory-store-hostname
Deleted SSM parameter: memory-store-id
Deleted SSM parameter: memory-store-arn
Deleted SSM parameter: memory-store-name
Deleted SSM parameter: memory-store-encryption_policy
Deleted SSM parameter: memory-store-network_policy
Deleted SSM parameter: memory-store-access_policy
Removed .env file: /home/sagemaker-user/samples/01-tutorials/01-fundamentals/07-memory-persistent-agents/.env
